<a href="https://colab.research.google.com/github/Sunidhishree/flyrank-ml-internship1/blob/main/work/notebooks/w04_signal_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Sunidhishree/flyrank-ml-internship1"
REPO_DIR = "flyrank-ml-internship1"
if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
valid = df[df["avg_position"] > 0].copy()
print(df.shape)

(30000, 45)


## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

impressions_90d and ctr are both dramatically right-skewed (skew 11.4 and 17.4 respectively) — a small number of very high-traffic, high-CTR outliers pull the mean far above the median (mean impressions 5,200 vs. median just 731; mean CTR 0.51% vs. median 0.07%). avg_position and days_since_last_update are moderately skewed (skew ~2 and ~1.2). word_count has notable missingness — only 22,301 of 30,000 rows have a value, consistent with the data dictionary's warning that missingness follows content_type. Any bucket analysis here needs medians, not means, or the heavy tail will distort the picture.

In [2]:
cols_to_check = ["impressions_90d", "ctr", "avg_position", "days_since_last_update", "word_count"]
print(df[cols_to_check].describe())

print("\nSkew (heavy tail check, >1 = notably right-skewed):")
print(df[cols_to_check].skew(numeric_only=True))

       impressions_90d           ctr  avg_position  days_since_last_update  \
count     30000.000000  30000.000000   30000.00000            30000.000000   
mean       5200.366300      0.510733      16.34238               46.098300   
std       16838.019547      3.279162      15.21679               42.078709   
min           1.000000      0.000000       0.00000                1.000000   
25%          81.000000      0.000000       6.20000               20.000000   
50%         731.000000      0.070000      10.80000               20.000000   
75%        3615.250000      0.290000      22.30000              104.000000   
max      517715.000000    100.000000     245.00000              373.000000   

         word_count  
count  22301.000000  
mean    3107.760325  
std     1452.382598  
min        8.000000  
25%     2413.000000  
50%     2877.000000  
75%     3666.000000  
max     9546.000000  

Skew (heavy tail check, >1 = notably right-skewed):
impressions_90d           11.384919
ctr       

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [3]:
bins = [0, 30, 90, 180, df["days_since_last_update"].max()]
labels = ["0-30d", "31-90d", "91-180d", "181d+"]
df["staleness_bucket"] = pd.cut(df["days_since_last_update"], bins=bins, labels=labels, include_lowest=True)
t1 = df.groupby("staleness_bucket").agg(n=("is_declining_label","size"), decline_rate=("is_declining_label","mean")).reset_index()
t1

/tmp/ipykernel_1328/699376414.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  t1 = df.groupby("staleness_bucket").agg(n=("is_declining_label","size"), decline_rate=("is_declining_label","mean")).reset_index()


,staleness_bucket,n,decline_rate
0,0-30d,20480,0.511377
1,31-90d,175,0.588571
2,91-180d,9171,0.611057
3,181d+,174,0.471264


Verdict: MIXED. Decline rate rises from 51.1% (0-30d) to 61.1% (91-180d) — supporting the staleness hypothesis — but then drops to 47.1% at 181d+, the lowest of all four buckets. This isn't the clean monotonic pattern the rule assumes. Also worth flagging: the 31-90d and 181d+ buckets have very small samples (n=175 and n=174) compared to the 0-30d bucket (n=20,480), so those two buckets' numbers are much less reliable — the "confirmation" in the middle buckets and the "reversal" at the tail could both partly be sample-size noise rather than a real trend reversal.

In [4]:
wc_bins = [0, 500, 1500, 3000, df["word_count"].max()]
wc_labels = ["<500", "500-1500", "1500-3000", "3000+"]
df["wc_bucket"] = pd.cut(df["word_count"], bins=wc_bins, labels=wc_labels, include_lowest=True)
t2 = df.groupby("wc_bucket").agg(n=("ctr","size"), avg_ctr=("ctr","mean")).reset_index()
t2

/tmp/ipykernel_1328/196950075.py:4: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  t2 = df.groupby("wc_bucket").agg(n=("ctr","size"), avg_ctr=("ctr","mean")).reset_index()


,wc_bucket,n,avg_ctr
0,<500,3,0.000000
1,500-1500,3225,2.089042
2,1500-3000,9511,0.404152
3,3000+,9562,0.289786


Verdict: OPPOSITE. CTR actually decreases as word count increases — from 2.09% (500-1500 words) down to 0.40% (1500-3000) down to 0.29% (3000+). This runs opposite to any assumption that longer content performs better on CTR. (The <500 bucket has only 3 rows and should be ignored — too small to mean anything.) This is a genuinely useful negative finding: word count is not a good proxy for click appeal in this data, and a rule that assumed "longer = better" would be actively wrong.

In [5]:
imp_bins = [0, df["impressions_90d"].quantile(0.25), df["impressions_90d"].quantile(0.5),
            df["impressions_90d"].quantile(0.75), df["impressions_90d"].max()]
imp_labels = ["Q1 (low)", "Q2", "Q3", "Q4 (high)"]
df["imp_bucket"] = pd.cut(df["impressions_90d"], bins=imp_bins, labels=imp_labels, include_lowest=True)
t3 = df.groupby("imp_bucket").agg(n=("is_declining_label","size"), decline_rate=("is_declining_label","mean")).reset_index()
t3

/tmp/ipykernel_1328/86870154.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  t3 = df.groupby("imp_bucket").agg(n=("is_declining_label","size"), decline_rate=("is_declining_label","mean")).reset_index()


,imp_bucket,n,decline_rate
0,Q1 (low),7503,0.376116
1,Q2,7499,0.604614
2,Q3,7498,0.625634
3,Q4 (high),7500,0.562000


Verdict: MIXED. Decline rate rises sharply from Q1 (37.6%) to Q2 (60.5%) and Q3 (62.6%), but then falls back to 56.2% at Q4 (highest-traffic pages). So the highest-traffic pages are not the most likely to be declining — mid-traffic pages are. This matters directly for my baseline rule from ML-07, which scores purely by raw impressions: it's implicitly assuming "more traffic = more urgency," but this test shows that assumption doesn't hold cleanly at the very top of the traffic range.

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [6]:
# CTR-vs-position, behind the CTR-fix flag logic
pos_bins = [0, 3, 10, 20, valid["avg_position"].max()]
pos_labels = ["1-3", "4-10", "11-20", "21+"]
valid["position_bucket"] = pd.cut(valid["avg_position"], bins=pos_bins, labels=pos_labels, include_lowest=True)
flag_test = valid.groupby("position_bucket").agg(n=("ctr","size"), avg_ctr=("ctr","mean")).reset_index()
flag_test

/tmp/ipykernel_1328/3374453306.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  flag_test = valid.groupby("position_bucket").agg(n=("ctr","size"), avg_ctr=("ctr","mean")).reset_index()


,position_bucket,n,avg_ctr
0,1-3,1141,2.714303
1,4-10,11842,0.651045
2,11-20,7273,0.323443
3,21+,8539,0.211333


Verdict: CONFIRMED. This is the cleanest, most monotonic result in the whole audit — average CTR drops steadily from 2.71% at position 1-3, to 0.65% at 4-10, to 0.32% at 11-20, to 0.21% at 21+. This directly supports the assumption behind FlyRank's CTR-fix flag: better position genuinely correlates with better CTR in this data, with no reversals or noise at any bucket.

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

A content team can trust the CTR-fix flag's core logic with confidence — position and CTR move together cleanly and consistently. Staleness and raw traffic volume are much weaker, noisier signals than they're often treated as: staleness shows a real relationship in the middle of its range but reverses at the extremes, and traffic volume alone is a poor proxy for decline urgency, since mid-traffic pages decline more than top-traffic pages do. These are observed, directional patterns in this specific dataset — not causal proof that fixing any one signal will move another, and any rule built purely on staleness or raw traffic (like my ML-07 baseline) should be treated as a rough first pass, not a final answer.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.